In [169]:
pip install pdfplumber

   ---------------------------------------- 0.0/5.6 MB ? eta -:--:--
   - -------------------------------------- 0.3/5.6 MB ? eta -:--:--
   --- ------------------------------------ 0.5/5.6 MB 985.5 kB/s eta 0:00:06
   ----- ---------------------------------- 0.8/5.6 MB 1.3 MB/s eta 0:00:04
   ----- ---------------------------------- 0.8/5.6 MB 1.3 MB/s eta 0:00:04
   ----- ---------------------------------- 0.8/5.6 MB 1.3 MB/s eta 0:00:04
   ------- -------------------------------- 1.0/5.6 MB 729.5 kB/s eta 0:00:07
   ------- -------------------------------- 1.0/5.6 MB 729.5 kB/s eta 0:00:07
   ------- -------------------------------- 1.0/5.6 MB 729.5 kB/s eta 0:00:07
   ------- -------------------------------- 1.0/5.6 MB 729.5 kB/s eta 0:00:07
   ------- -------------------------------- 1.0/5.6 MB 729.5 kB/s eta 0:00:07
   --------- ------------------------------ 1.3/5.6 MB 516.0 kB/s eta 0:00:09
   ------------- -------------------------- 1.8/5.6 MB 662.3 kB/s eta 0:00:06
   -------

In [172]:
pip install langchain



  Using cached tenacity-9.1.2-py3-none-any.whl.metadata (1.2 kB)
  Using cached langgraph_checkpoint-3.0.1-py3-none-any.whl.metadata (4.7 kB)
  Using cached requests_toolbelt-1.0.0-py2.py3-none-any.whl.metadata (14 kB)
Using cached langgraph_checkpoint-3.0.1-py3-none-any.whl (46 kB)
Using cached tenacity-9.1.2-py3-none-any.whl (28 kB)
Using cached requests_toolbelt-1.0.0-py2.py3-none-any.whl (54 kB)

   --- ------------------------------------  1/12 [tenacity]
   ------------- --------------------------  4/12 [requests-toolbelt]
   ------------- --------------------------  4/12 [requests-toolbelt]
   ------------- --------------------------  4/12 [requests-toolbelt]
   ---------------- -----------------------  5/12 [langsmith]
   ---------------- -----------------------  5/12 [langsmith]
   ---------------- -----------------------  5/12 [langsmith]
   ---------------- -----------------------  5/12 [langsmith]
   ---------------- -----------------------  5/12 [langsmith]
   -----------

pdf reader et pdf chat

                        PDF
                        │
                        ▼
                        Extraction du contenu (texte / structure)
                        │
                        ▼
                        Indexation (chunks + embeddings)
                        │
                        ▼
                        RAG
                        │
                        ▼
                        LLM
                        │
                        ▼
                        Réponses conversationnelles sur le document


In [177]:
# --- Import robuste pour LangChain Splitter ---
try:
    from langchain.text_splitter import RecursiveCharacterTextSplitter
    print("✔ Import depuis langchain.text_splitter")
except ImportError:
    try:
        from langchain_text_splitters import RecursiveCharacterTextSplitter
        print("✔ Import depuis langchain_text_splitters")
    except ImportError:
        try:
            from langchain_core.text_splitter import RecursiveCharacterTextSplitter
            print("✔ Import depuis langchain_core.text_splitter")
        except ImportError:
            raise ImportError(
                "❌ Impossible d'importer RecursiveCharacterTextSplitter.\n"
                "Installe une version récente :\n"
                "pip install -U langchain langchain-community langchain-core"
            )


✔ Import depuis langchain_text_splitters


In [179]:
!pip install -U langchain langchain-core langchain-community



In [180]:
import langchain
print(langchain.__version__)


1.1.3


In [181]:
import pdfplumber
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.embeddings import HuggingFaceEmbeddings
from langchain_community.vectorstores import FAISS
from langchain_core.prompts import ChatPromptTemplate
import httpx
from openai import OpenAI
import asyncio
import os


In [182]:
def extract_text_from_pdfs(pdf_paths):
    """
    pdf_paths : liste de chemins PDF
    """
    text = ""
    for pdf in pdf_paths:
        with pdfplumber.open(pdf) as doc:
            for page in doc.pages:
                content = page.extract_text()
                if content:
                    text += content + "\n"
    return text.strip()


In [193]:
def create_vector_store(text):
    """
    Transforme le texte PDF en chunks + embeddings + FAISS retriever
    """

    chunks = RecursiveCharacterTextSplitter(
        chunk_size=8000,
        chunk_overlap=1000
    ).split_text(text)

    embeddings = HuggingFaceEmbeddings(
        model_name="sentence-transformers/all-MiniLM-L6-v2"
    )

    return FAISS.from_texts(chunks, embeddings)


In [184]:
def build_medical_prompt(question, context):
    template = """
Tu es un assistant médical intelligent.

RÔLE :
- Si le document contient des résultats médicaux : explique leur signification.
- Si c’est une ordonnance : identifie les médicaments et leur usage.
- Si c’est un rapport : résume les conclusions.
- Si le texte est administratif : explique les démarches.
- Si le texte est incomplet : donne la meilleure interprétation possible.

⚠️ RÈGLES :
- Pas de spéculation.
- Pas d’invention médicale.
- Utilise uniquement le CONTEXTE fourni.
- Si l’information manque → dire "Donnée absente du document".

📘 CONTEXTE :
{context}

❓ QUESTION :
{question}

💬 RÉPONSE :
"""

    prompt = ChatPromptTemplate.from_template(template)
    return prompt.format(context=context, question=question)


In [185]:
def call_esprit_llm(prompt, esprit_api_key):
    http_client = httpx.Client(verify=False)

    client = OpenAI(
        api_key=esprit_api_key,
        base_url="https://tokenfactory.esprit.tn/api",
        http_client=http_client
    )

    response = client.chat.completions.create(
        model="hosted_vllm/Llama-3.1-70B-Instruct",
        messages=[
            {"role": "system", "content": "Tu es un assistant médical et administratif utile et concis."},
            {"role": "user", "content": prompt}
        ],
        temperature=0.4,
        max_tokens=700,
        top_p=0.9
    )

    return response.choices[0].message.content


In [186]:
def analyse_pdf_chat(question, pdf_paths, esprit_api_key):
    # Extraction du texte PDF
    text = extract_text_from_pdfs(pdf_paths)
    if not text:
        return "❌ Aucun texte détecté dans les PDF."

    # Création du store FAISS
    vector_store = create_vector_store(text)

    retriever = vector_store.as_retriever(search_kwargs={"k": 5})
    docs = retriever.invoke(question)

    if not docs:
        return "❌ Aucun passage pertinent trouvé."

    # Context
    context = "\n\n".join([d.page_content for d in docs])

    # Prompt intelligent
    prompt = build_medical_prompt(question, context)

    # Appel API
    return call_esprit_llm(prompt, esprit_api_key)


In [ ]:
esprit_api_key = ""

question = "Explique les résultats de cette ordonnance."
pdf_paths = [""]

response = analyse_pdf_chat(question, pdf_paths, esprit_api_key)
print("\n🧠 RÉPONSE :\n")
print(response)



C:\Users\hp\AppData\Local\Temp\ipykernel_25920\3753340387.py:11: LangChainDeprecationWarning: The class `HuggingFaceEmbeddings` was deprecated in LangChain 0.2.2 and will be removed in 1.0. An updated version of the class exists in the `langchain-huggingface package and should be used instead. To use it run `pip install -U `langchain-huggingface` and import as `from `langchain_huggingface import HuggingFaceEmbeddings``.
  embeddings = HuggingFaceEmbeddings(



🧠 RÉPONSE :

Les résultats des analyses médicales de Mlle ZAHROUNI MAYA sont présentés dans ce document. Voici une explication détaillée des résultats :

**Cyto-hématologie**

* Le nombre de globules rouges est de 4,53 millions/mm³, ce qui est dans la plage normale (4-5,5 millions/mm³).
* L'hématocrite est de 40%, ce qui est également dans la plage normale (35-47%).
* L'hémoglobine est de 13,8 g%, ce qui est dans la plage normale (12-15 g%).
* Le volume globulaire moyen est de 87 µm³, ce qui est dans la plage normale (82-92 µm³).
* Le taux globulaire moyen hb est de 30,5 Pg/GR, ce qui est dans la plage normale (27-32 Pg/GR).
* La concentration globulaire moyenne est de 34,9%, ce qui est dans la plage normale (31-36%).
* Le RDW-CV est de 12,7%, ce qui est dans la plage normale (11-15%).

**Leucocytes**

* Le nombre de leucocytes est de 5 860/mm³, ce qui est dans la plage normale (4 000-10 000/mm³).
* La formule leucocytaire montre :
 + Les neutrophiles représentent 55% des leucocytes, 

In [194]:
import pdfplumber
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.embeddings import HuggingFaceEmbeddings
from langchain_community.vectorstores import FAISS

# --- Extraire texte depuis un PDF ---
def extract_text_from_pdf(path):
    text = ""
    with pdfplumber.open(path) as pdf:
        for page in pdf.pages:
            page_text = page.extract_text()
            if page_text:
                text += page_text + "\n"
    return text.strip()

# --- Construire le vector store ---
def create_vector_store_from_text(text):
    splitter = RecursiveCharacterTextSplitter(
        chunk_size=2000,
        chunk_overlap=300
    )
    
    chunks = splitter.split_text(text)
    
    embedding_model = HuggingFaceEmbeddings(
        model_name="sentence-transformers/all-MiniLM-L6-v2"
    )
    
    vector_store = FAISS.from_texts(chunks, embedding_model)
    return vector_store


In [ ]:
pdf_path = r""

# 1. Extraction texte
pdf_text = extract_text_from_pdf(pdf_path)

# 2. Construction vector store
vector_store = create_vector_store_from_text(pdf_text)

print("✔ vector_store créé avec succès")


✔ vector_store créé avec succès


In [196]:
def xai_self_check(question, answer, context, esprit_api_key):
    prompt = f"""
Tu es un expert en évaluation RAG et sécurité médicale.
Analyse la réponse du modèle STRICTEMENT selon le contexte fourni.

Donne un JSON STRICT avec les champs suivants :

{{
 "faithfulness": 0,
 "relevance": 0,
 "completeness": 0,
 "risk": false,
 "errors": [],
 "final_score": 0
}}

Définitions :

- "faithfulness" = 1 si la réponse correspond STRICTEMENT au contexte PDF, sinon 0.
- "relevance" = score 0 à 100 de pertinence par rapport à la question.
- "completeness" = 1 si la réponse inclut toutes les infos du contexte nécessaires.
- "risk" = true si la réponse peut être dangereuse, fausse, ou spéculative.
- "errors" = liste détaillée des hallucinations ou contradictions.
- "final_score" = score pondéré :  
      50% faithfulness + 30% relevance + 20% completeness.

📘 CONTEXTE PDF :
{context}

❓ QUESTION :
{question}

💬 RÉPONSE DU MODÈLE :
{answer}

Répond uniquement en JSON.
    """

    http_client = httpx.Client(verify=False)
    client = OpenAI(
        api_key=esprit_api_key,
        base_url="https://tokenfactory.esprit.tn/api",
        http_client=http_client
    )

    response = client.chat.completions.create(
        model="hosted_vllm/Llama-3.1-70B-Instruct",
        messages=[{"role": "user", "content": prompt}],
        temperature=0,
        max_tokens=500
    )

    raw = response.choices[0].message.content.strip()

    # nettoyage fallback
    try:
        return json.loads(raw)
    except:
        print("⚠️ JSON invalide retourné, affichage brut :")
        print(raw)
        return {
            "faithfulness": 0,
            "relevance": 0,
            "completeness": 0,
            "risk": True,
            "errors": ["Invalid JSON from LLM", raw],
            "final_score": 0
        }


In [197]:
def analyse_pdf_chat(question, pdf_text, vector_store, esprit_api_key):
    # Retrieve context from vector store
    try:
        docs = vector_store.similarity_search(question, k=3)
        context = "\n\n".join([d.page_content for d in docs])
    except:
        context = pdf_text

    # Build prompt
    prompt = f"""
Tu es un assistant médical intelligent.
Analyse ces informations :

📘 CONTEXTE :
{context}

❓ QUESTION :
{question}

Réponse claire et concise :
"""

    # Call ESPRIT API
    http_client = httpx.Client(verify=False)
    client = OpenAI(
        api_key=esprit_api_key,
        base_url="https://tokenfactory.esprit.tn/api",
        http_client=http_client
    )

    response = client.chat.completions.create(
        model="hosted_vllm/Llama-3.1-70B-Instruct",
        messages=[{"role": "user", "content": prompt}],
        temperature=0.4,
        max_tokens=400
    )

    answer = response.choices[0].message.content.strip()

    return answer, context


In [198]:
def test_and_evaluate(question, pdf_text, vector_store, esprit_api_key):
    print("\n==============================")
    print(f"🧠 QUESTION : {question}")
    print("==============================")

    # --- 1. Get RAG answer
    answer, context = analyse_pdf_chat(question, pdf_text, vector_store, esprit_api_key)

    print("\n🤖 RÉPONSE DU MODÈLE :\n")
    print(answer)

    # --- 2. Evaluate answer with XAI
    evaluation = xai_self_check(question, answer, context, esprit_api_key)

    print("\n📊 RAPPORT XAI :")
    print(evaluation)

    return {
        "question": question,
        "answer": answer,
        "context": context,
        "evaluation": evaluation
    }


In [ ]:
esprit_api_key = ""

question = "Explique les résultats de cette ordonnance."

result = test_and_evaluate(question, pdf_text, vector_store, esprit_api_key)

print("\n🎉 RÉSUMÉ FINAL :")
print(result)



🧠 QUESTION : Explique les résultats de cette ordonnance.

🤖 RÉPONSE DU MODÈLE :

Analysons les résultats de cette ordonnance médicale :

**Cytologie et hématologie :**

* Les globules rouges (4,53 millions/mm³) et l'hémoglobine (13,8 g/dL) sont légèrement élevés, ce qui peut indiquer une augmentation de la production de globules rouges.
* Le taux de globules blancs (5 860/mm³) est légèrement élevé, ce qui peut indiquer une infection ou une inflammation.
* La formule leucocytaire montre une augmentation des neutrophiles (55 %) et des lymphocytes (39 %), ce qui peut indiquer une réponse immunitaire à une infection.

**Biochimie :**

* La créatinine (8,8 mg/L) est légèrement élevée, ce qui peut indiquer une altération de la fonction rénale.
* Le fer sérique (1,65 mg/L) est légèrement élevé, ce qui peut indiquer une augmentation de l'absorption de fer.
* La calcémie (101 mg/L) est normale.
* Le magnésium plasmatique (22 mg/L) est légèrement élevé, ce qui peut indiquer une augmentation de 

In [200]:
import pdfplumber
import json
import httpx
from openai import OpenAI

from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.embeddings import HuggingFaceEmbeddings
from langchain_community.vectorstores import FAISS

import pandas as pd
import numpy as np


In [201]:
def extract_text_from_pdf(path):
    text = ""
    with pdfplumber.open(path) as pdf:
        for page in pdf.pages:
            t = page.extract_text()
            if t:
                text += t + "\n"
    return text.strip()


In [202]:
def create_vector_store(text):
    splitter = RecursiveCharacterTextSplitter(
        chunk_size=1500,
        chunk_overlap=150
    )
    chunks = splitter.split_text(text)

    embeddings = HuggingFaceEmbeddings(
        model_name="sentence-transformers/all-MiniLM-L6-v2"
    )

    vs = FAISS.from_texts(chunks, embeddings)
    return vs


In [203]:
def build_medical_analysis_prompt(question, context):

    return f"""
Tu es un assistant médical ultra-fiable spécialisé dans l’analyse de documents biologiques.

Tu dois répondre STRICTEMENT à partir du texte fourni.
Aucune connaissance extérieure n’est autorisée.

────────────────────────────────────
📄 CONTEXTE (Données du document)
────────────────────────────────────
{context}

────────────────────────────────────
⚠️ RÈGLES ANTI-HALLUCINATION
────────────────────────────────────
1. Ne JAMAIS interpréter sans comparer aux valeurs normales présentes dans le document.
2. Ne JAMAIS écrire des phrases comme : “peut indiquer”, “probablement”, “infection”, “inflammation”.
3. Ne PAS inventer de diagnostic ou de maladie.
4. Si une information n'est pas dans le document → écrire : “Non indiqué dans le document.”
5. Réponse courte, factuelle et maximum 8 lignes.
6. Ne pas réécrire tout le document.

────────────────────────────────────
❓ QUESTION UTILISATEUR
{question}

────────────────────────────────────
🧪 RÉPONSE FACTUELLE (basée EXCLUSIVEMENT sur le document) :
"""


In [204]:
def ask_llm(prompt):
    response = client.chat.completions.create(
        model="hosted_vllm/Llama-3.1-70B-Instruct",
        messages=[
            {"role": "system", "content": "Assistant médical strictement factuel."},
            {"role": "user", "content": prompt}
        ],
        max_tokens=500,
        temperature=0.0
    )
    return response.choices[0].message.content


In [205]:
def evaluate_answer(context, answer):
    errors = []

    # FAITHFULNESS = vérifier si les phrases ne sortent pas du contexte
    faithfulness = 100 if answer.lower() in context.lower() or any(w in context.lower() for w in answer.lower().split()) else 0

    # RELEVANCE = question pertinente par rapport au document
    relevance = 100 if len(set(answer.split()) & set(context.split())) > 5 else 60

    # COMPLETENESS = la réponse couvre-t-elle les valeurs évoquées
    completeness = 100 if "mm" in answer or "g/l" in answer or "%" in answer else 0

    # RISK = contient-il des mots médicaux dangereux ?
    risky_words = ["infection", "inflammation", "maladie", "insuffisance", "risque", "grave"]
    risk = any(w in answer.lower() for w in risky_words)

    return {
        "faithfulness": faithfulness,
        "relevance": relevance,
        "completeness": completeness,
        "risk": risk,
        "errors": errors,
        "final_score": (faithfulness*0.4 + relevance*0.3 + completeness*0.3)
    }


In [206]:
def analyse_pdf_chat(question, pdf_path):
    text = extract_text_from_pdf(pdf_path)
    vector_store = create_vector_store(text)

    docs = vector_store.similarity_search(question, k=4)
    context = "\n\n".join([d.page_content for d in docs])

    prompt = build_medical_analysis_prompt(question, context)
    answer = ask_llm(prompt)

    eval_report = evaluate_answer(context, answer)

    return {
        "question": question,
        "answer": answer,
        "context_used": context,
        "evaluation": eval_report
    }


In [ ]:
pdf_path = r""
question = "Explique les résultats de cette ordonnance."

result = analyse_pdf_chat(question, pdf_path)

print("\n🧠 RÉPONSE :\n", result["answer"])
print("\n📊 RAPPORT XAI :\n", result["evaluation"])



🧠 RÉPONSE :
 Les résultats de l'analyse sanguine de Mlle ZAHROUNI MAYA sont les suivants :

- Numération et formule sanguine : 
  - Globules rouges : 4,53 10^6/mm^3 (normale)
  - Leucocytes : 5 860/mm^3 (normale)
  - Plaquettes : 384 000/mm^3 (normale)

- Biochimie :
  - Créatinine : 8,8 mg/l (normale)
  - Fer sérique : 1,65 mg/l (anormale, supérieure à la valeur de référence)
  - Calciémie : 101 mg/l (normale)
  - Magnésium plasmatique : 22 mg/l (normale)
  - T.G.O (ASAT à 37°) : 20 U/l (normale)
  - T.G.P (ALAT à 37°) : 16 U/l (normale)

Les autres résultats sont également dans les valeurs de référence normales.

📊 RAPPORT XAI :
 {'faithfulness': 100, 'relevance': 100, 'completeness': 100, 'risk': False, 'errors': [], 'final_score': 100.0}


In [209]:
questions = [
    "Explique les globules rouges.",
    "Y a-t-il des anomalies dans la NFS ?",
    "La créatinine est-elle normale ?",
    "Résumé de toute l’analyse."
]

for q in questions:
    print("\n==============================")
    print("🧠 QUESTION :", q)
    print("==============================")

    result = analyse_pdf_chat(q, pdf_path)

    print("\n🤖 RÉPONSE :\n", result["answer"])
    print("\n📊 XAI :\n", result["evaluation"])



🧠 QUESTION : Explique les globules rouges.

🤖 RÉPONSE :
 Les globules rouges ont une valeur de 4,53 10^6/mm^3, ce qui est compris entre les valeurs de référence de 4 - 5,5.
L'hématocrite est de 40 %, compris entre 35 - 47 %.
L'hémoglobine est de 13,80 g %, comprise entre 12 - 15 g %.
Le volume globulaire moyen est de 87 µ.cube, compris entre 82 - 92 µ.cube.
Le taux globulaire moyen hb est de 30,5 Pg/GR, compris entre 27 - 32 Pg/GR.
La concentration globulaire moyenne est de 34,9 %, comprise entre 31 - 36 %.
Le RDW-CV est de 12,7 %, compris entre 11 - 15 %.

📊 XAI :
 {'faithfulness': 100, 'relevance': 100, 'completeness': 100, 'risk': False, 'errors': [], 'final_score': 100.0}

🧠 QUESTION : Y a-t-il des anomalies dans la NFS ?

🤖 RÉPONSE :
 Non, les valeurs de la numération formule sanguine (NFS) sont dans les plages de référence normales.

📊 XAI :
 {'faithfulness': 100, 'relevance': 60, 'completeness': 0, 'risk': False, 'errors': [], 'final_score': 58.0}

🧠 QUESTION : La créatinine es